In [2]:
import pandas as pd
import gensim
import pickle
import re
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import os

print("🚀 Memulai proses Training Model Hibrida (FastText + SVM)...")

# 1. BACA DATASET
df = pd.read_csv('dataset_chat.csv')

# 2. PRAPEMROSESAN TEKS (Membersihkan simbol)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text) # Hapus tanda baca
    return text.strip()

df['teks_bersih'] = df['teks_chat'].apply(clean_text)

# 3. TRAINING FASTTEXT (Ekstraksi Fitur / Word Embedding)
# FastText butuh data teks mentah dalam bentuk file .txt untuk belajar
df['teks_bersih'].to_csv('temp_corpus.txt', index=False, header=False)

print("🧠 Melatih FastText (Memahami sub-word dan typo)...")
# Melatih model Unsupervised FastText
ft_model = gensim.models.FastText.load_model('temp_corpus.txt', model='skipgram', dim=100)
ft_model.save_model("fasttext_model.bin")
os.remove('temp_corpus.txt') # Hapus file temp

# Fungsi untuk mengubah kalimat menjadi angka (Vektor)
def get_sentence_vector(text):
    return ft_model.get_sentence_vector(text)

# Ubah semua teks di CSV menjadi Matrix Vektor
X = np.array(df['teks_bersih'].apply(get_sentence_vector).tolist())
y = df['label_intent']

# 4. TRAINING SVM DENGAN PLATT SCALING
print("⚖️ Melatih SVM (Mencari Hyperplane & Platt Scaling)...")
# probability=True adalah kunci utama "Platt Scaling" di skripsimu!
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X, y)

# Evaluasi sederhana
y_pred = svm_model.predict(X)
print("\n📊 Hasil Akurasi Training (Confusion Matrix):")
print(classification_report(y, y_pred))

# 5. SIMPAN MODEL SVM
with open('svm_model.pkl', 'wb') as file:
    pickle.dump(svm_model, file)

print("✅ Selesai! Model berhasil disimpan sebagai 'fasttext_model.bin' dan 'svm_model.pkl'")

🚀 Memulai proses Training Model Hibrida (FastText + SVM)...
🧠 Melatih FastText (Memahami sub-word dan typo)...


AttributeError: type object 'FastText' has no attribute 'load_model'